In [1]:
import glob
import os

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.util import compare_images
from skimage import io, color, measure
from skimage.measure import regionprops
from skimage.measure import label as sk_label  # Renaming the label function to avoid conflicts

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
# Import images

images_dir = '/Volumes/bamfaile/Franklin/20241021/ACTB_KI_NGN2_Batch20241011_D10/Puro10min/50ms/Selected_for_analysis'
images_path = os.path.join(images_dir,'*.stk') 
images_files = np.sort(glob.glob(images_path))

# Extract the integers before the first underscore in the filenames
loaded_image_ids = {
    os.path.basename(f).split('_')[0]  # Extract the part before the first underscore
    for f in images_files
}

masks_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min/ROIs/Cytoplasm'
masks_path = os.path.join(masks_dir,'*.tif') 
masks_files = np.sort(glob.glob(masks_path))

# Filter the new images based on matching IDs
mask_files = [
    f for f in masks_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
]

print(len(images_files))
print(len(mask_files))

In [ ]:
# Read images into list

images = []
masks = []

for file in tqdm(images_files):
    image = imread(file)
    images.append(image)

for file in tqdm(mask_files):
    image = imread(file)
    masks.append(image)

In [ ]:
# Create copies of cytoplasm masks for manipulation in Napari

mask_cytoplasm_copy = masks.copy()
plt.imshow(mask_cytoplasm_copy[3])

In [5]:
import napari

In [72]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [109]:
# Adds images to Napari viewer

for i, img in enumerate(images):
    layer_name = f'image_{i}'  # Dynamically name each layer
    viewer.add_image(img, name=layer_name)

In [110]:
# Adds masks to Napari viewer

for i, img in enumerate(mask_cytoplasm_copy):
    layer_name = f'segmentation_{i}'  # Dynamically name each layer
    labels_layer = viewer.add_labels(img, name=layer_name, opacity=0.6)

In [ ]:
plt.imshow(mask_cytoplasm_copy[4])

In [111]:
# Save cytoplasm masks

mask_cytoplasm_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min/ROIs_corrected'


In [ ]:
# Save mask images

# Save mask images
for img, tiff_file in zip(mask_cytoplasm_copy, mask_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    
    # Replace spaces with underscores
    file_name = file_name.replace(' ', '_')
    
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(mask_cytoplasm_dir, file_name + '.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img)

In [ ]:
mask_files

In [ ]:
len(mask_files)